# 02D v2 — Resumable Biking pose extraction (safer full-video run)

This notebook improves the previous full-video attempt:

- Uses the already-trained ViTPose checkpoint; **no training occurs**.
- Uses CPU inference for this diagnostic run because the earlier long extraction caused kernel restarts. CPU use is a diagnostic choice, not proof the GPU is faulty.
- Reads the video in order and processes every 4th original frame.
- Tries YOLO confidence `0.25` first, and retries at `0.10` **only if the first attempt found no person**.
- Processes at most 5 new sampled frames per run, then saves a resumable partial NPZ. Re-run the batch cell until it reports completion.
- Saves one JPEG overlay per sampled frame and separates detection success from pose success.
- Does not use ByteTrack yet and does not encode an MP4. It chooses the largest detected person independently in each frame, so identity consistency is not guaranteed.

**Important:** upload this notebook into `C:\Users\admin\ViTPose_Action_Recognition\notebooks`. Keep the previous notebook and previous output files unchanged.

Run cells in order, one at a time. Do not use Run All.

## Cell 1 — Imports, project paths, checkpoint and detector

In [1]:
from pathlib import Path
import sys
import gc
import time
import json
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose,
    get_affine_transform,
    decode_heatmaps,
    draw_skeleton_on,
    IMG_H,
    IMG_W,
    IMG_MEAN,
    IMG_STD,
)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"

assert CHECKPOINT_PATH.is_file(), f"ViTPose checkpoint not found: {CHECKPOINT_PATH}"
assert YOLO_PATH.is_file(), f"YOLO weights not found: {YOLO_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

# CPU-only diagnostic run: reduces GPU/CUDA factors while debugging extraction.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
SAVED_EPOCH = checkpoint.get("epoch", "unknown")
del checkpoint
gc.collect()
print("ViTPose loaded. Saved epoch:", SAVED_EPOCH, flush=True)

print("Loading YOLO detector...", flush=True)
detector = YOLO(str(YOLO_PATH))
print("YOLO loaded. Device:", DEVICE, flush=True)

Loading ViTPose checkpoint on CPU...
ViTPose loaded. Saved epoch: 209
Loading YOLO detector...
YOLO loaded. Device: cpu


## Cell 2 — Select Biking video and configure the run

This version writes to new `_v2` output paths to avoid mixing results with the previous attempt. It processes 5 new sampled frames each time the batch cell is run.

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
biking = manifest[manifest["label"].astype(str).str.casefold().eq("biking")].reset_index(drop=True)
assert len(biking) > 0, "No Biking videos found in manifest."

# Use the same first Biking video as the previous pilot.
VIDEO_PATH = Path(str(biking.loc[0, "video_path"]))
VIDEO_LABEL = str(biking.loc[0, "label"])
assert VIDEO_PATH.is_file(), f"Video not found: {VIDEO_PATH}"

FRAME_STEP = 4
DETECTION_CONF = 0.25
DETECTION_CONF_FALLBACK = 0.10
YOLO_IMGSZ = 640
POSE_THRESH = 0.10
BATCH_SAMPLES_PER_RUN = 5

FRAME_DIR = PROJECT_DIR / "outputs" / "pose_videos" / "biking_full_video_frames_v2"
KEYPOINT_DIR = PROJECT_DIR / "data" / "processed" / "keypoints_single_tracked"
FRAME_DIR.mkdir(parents=True, exist_ok=True)
KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)

PARTIAL_NPZ = KEYPOINT_DIR / "biking_full_video_partial_v2.npz"
FINAL_NPZ = KEYPOINT_DIR / "biking_full_video_keypoints_v2.npz"
REPORT_CSV = PROJECT_DIR / "data" / "processed" / "biking_full_video_report_v2.csv"

print("Label:", VIDEO_LABEL)
print("Video:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.is_file())
print("Frame sampling interval:", FRAME_STEP)
print("Primary YOLO confidence:", DETECTION_CONF)
print("Fallback YOLO confidence:", DETECTION_CONF_FALLBACK)
print("Frames per batch run:", BATCH_SAMPLES_PER_RUN)
print("Frame output folder:", FRAME_DIR)
print("Partial file:", PARTIAL_NPZ)
print("Final file:", FINAL_NPZ)

if FINAL_NPZ.exists():
    print("WARNING: final v2 output already exists. This notebook will not automatically overwrite it.")

Label: Biking
Video: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports\Biking\v_Biking_g01_c01.avi
Video exists: True
Frame sampling interval: 4
Primary YOLO confidence: 0.25
Fallback YOLO confidence: 0.1
Frames per batch run: 5
Frame output folder: C:\Users\admin\ViTPose_Action_Recognition\outputs\pose_videos\biking_full_video_frames_v2
Partial file: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_single_tracked\biking_full_video_partial_v2.npz
Final file: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_single_tracked\biking_full_video_keypoints_v2.npz


## Cell 3 — Pose inference and safe NPZ-saving helpers

In [3]:
def pose_for_box_cpu(frame_rgb, bbox):
    """Estimate 17 keypoints for one person crop using the original notebook helpers."""
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = float(np.clip(x1, 0, w - 1))
    x2 = float(np.clip(x2, 0, w))
    y1 = float(np.clip(y1, 0, h - 1))
    y2 = float(np.clip(y2, 0, h))
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, trans, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()

    keypoints = decode_heatmaps(heatmaps, center, scale).astype(np.float32)
    del tensor, heatmaps, crop
    return keypoints if keypoints.shape == (17, 3) else None


def save_pose_npz(path, observations, label, video_path, fps, frame_step):
    """Persist the current observations so a later run can resume after a kernel restart."""
    if not observations:
        return
    np.savez_compressed(
        path,
        keypoints=np.stack([o["keypoints"] for o in observations]).astype(np.float32),
        boxes=np.stack([o["bbox"] for o in observations]).astype(np.float32),
        frame_indices=np.asarray([o["frame_index"] for o in observations], dtype=np.int32),
        detection_found=np.asarray([o["detection_found"] for o in observations], dtype=np.bool_),
        has_person=np.asarray([o["pose_ok"] for o in observations], dtype=np.bool_),
        confidence_used=np.asarray([o["confidence_used"] for o in observations], dtype=np.float32),
        fallback_used=np.asarray([o["fallback_used"] for o in observations], dtype=np.bool_),
        label=np.asarray(str(label)),
        video_path=np.asarray(str(video_path)),
        fps=np.asarray(float(fps), dtype=np.float32),
        frame_step=np.asarray(int(frame_step), dtype=np.int32),
    )


def load_observations(path):
    """Read a partial file into a list of per-frame records."""
    if not path.is_file():
        return []
    with np.load(path, allow_pickle=False) as d:
        kps = d["keypoints"].copy()
        boxes = d["boxes"].copy()
        indices = d["frame_indices"].copy()
        detected = d["detection_found"].copy()
        pose_ok = d["has_person"].copy()
        confs = d["confidence_used"].copy()
        fallback = d["fallback_used"].copy()
    out = []
    for i in range(len(indices)):
        out.append({
            "keypoints": kps[i],
            "bbox": boxes[i],
            "frame_index": int(indices[i]),
            "detection_found": bool(detected[i]),
            "pose_ok": bool(pose_ok[i]),
            "confidence_used": float(confs[i]),
            "fallback_used": bool(fallback[i]),
        })
    return out

print("Pose inference and resume helpers are ready.", flush=True)

Pose inference and resume helpers are ready.


## Cell 4 — Process the next batch of up to 5 sampled frames

Run this cell repeatedly until it prints `VIDEO COMPLETE`. Each sampled frame is saved immediately to the partial NPZ. If the kernel restarts, restart the kernel, run Cells 1–4 in order, then run this cell again; it will resume from the latest partial file. It skips already saved frame indices.

In [12]:
if FINAL_NPZ.is_file():
    print("Final output already exists:", FINAL_NPZ)
    print("Skipping extraction to avoid overwriting it. Inspect Cell 6 instead.", flush=True)
else:
    observations = load_observations(PARTIAL_NPZ)
    already_done = {o["frame_index"] for o in observations}
    last_saved_index = max(already_done) if already_done else -1
    print("Previously saved sampled frames:", len(observations), flush=True)
    print("Last saved original frame index:", last_saved_index, flush=True)

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    assert cap.isOpened(), f"Could not open video: {VIDEO_PATH}"
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    reported_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    print("FPS:", fps, "| reported total frames:", reported_total, flush=True)

    frame_index = 0
    new_samples_this_run = 0
    video_exhausted = False
    started = time.time()

    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                video_exhausted = True
                break

            is_sample = (frame_index % FRAME_STEP == 0)
            is_new = frame_index not in already_done
            if is_sample and is_new and frame_index > last_saved_index:
                print(f"Processing original frame {frame_index}...", flush=True)
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

                # Attempt primary YOLO confidence first.
                result = detector.predict(
                    frame_bgr,
                    classes=[0],
                    conf=DETECTION_CONF,
                    imgsz=YOLO_IMGSZ,
                    device="cpu",
                    verbose=False,
                )[0]
                confidence_used = DETECTION_CONF
                fallback_used = False

                # Retry only if the primary threshold finds no person.
                if result.boxes is None or len(result.boxes) == 0:
                    result = detector.predict(
                        frame_bgr,
                        classes=[0],
                        conf=DETECTION_CONF_FALLBACK,
                        imgsz=YOLO_IMGSZ,
                        device="cpu",
                        verbose=False,
                    )[0]
                    confidence_used = DETECTION_CONF_FALLBACK
                    fallback_used = True

                detection_found = result.boxes is not None and len(result.boxes) > 0
                bbox = np.full((4,), np.nan, dtype=np.float32)
                keypoints = np.full((17, 3), np.nan, dtype=np.float32)
                pose_ok = False
                annotated_rgb = frame_rgb.copy()

                if detection_found:
                    boxes = result.boxes.xyxy.cpu().numpy()
                    areas = (
                        np.maximum(0, boxes[:, 2] - boxes[:, 0]) *
                        np.maximum(0, boxes[:, 3] - boxes[:, 1])
                    )
                    bbox = boxes[int(np.argmax(areas))].astype(np.float32)
                    predicted = pose_for_box_cpu(frame_rgb, bbox)
                    if predicted is not None and predicted.shape == (17, 3):
                        keypoints = predicted
                        pose_ok = bool(np.isfinite(keypoints[:, :2]).all(axis=1).any())
                        if pose_ok:
                            annotated_rgb = draw_skeleton_on(
                                frame_rgb.copy(), keypoints, thresh=POSE_THRESH
                            )
                        x1, y1, x2, y2 = map(int, bbox)
                        cv2.rectangle(annotated_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        tag = f"Biking frame={frame_index} conf={confidence_used:.2f}"
                        cv2.putText(
                            annotated_rgb, tag, (5, 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1
                        )
                else:
                    cv2.putText(
                        annotated_rgb, f"NO PERSON frame={frame_index}", (5, 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1
                    )

                out_img = FRAME_DIR / f"frame_{frame_index:06d}.jpg"
                saved_img = cv2.imwrite(
                    str(out_img), cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
                )
                if not saved_img:
                    raise RuntimeError(f"Could not save image: {out_img}")

                observation = {
                    "keypoints": keypoints,
                    "bbox": bbox,
                    "frame_index": int(frame_index),
                    "detection_found": bool(detection_found),
                    "pose_ok": bool(pose_ok),
                    "confidence_used": float(confidence_used),
                    "fallback_used": bool(fallback_used),
                }
                observations.append(observation)
                already_done.add(frame_index)

                # Save partial after every sample so a crash loses little or no work.
                save_pose_npz(
                    PARTIAL_NPZ, observations, VIDEO_LABEL, VIDEO_PATH, fps, FRAME_STEP
                )

                print(
                    f"  detection={detection_found}; pose_ok={pose_ok}; "
                    f"fallback={fallback_used}; total_saved={len(observations)}",
                    flush=True,
                )

                del result, frame_rgb, annotated_rgb
                gc.collect()
                new_samples_this_run += 1

                if new_samples_this_run >= BATCH_SAMPLES_PER_RUN:
                    break

            frame_index += 1
    finally:
        cap.release()

    if video_exhausted:
        save_pose_npz(
            PARTIAL_NPZ, observations, VIDEO_LABEL, VIDEO_PATH, fps, FRAME_STEP
        )
        if observations:
            save_pose_npz(
                FINAL_NPZ, observations, VIDEO_LABEL, VIDEO_PATH, fps, FRAME_STEP
            )

        detection_count = sum(o["detection_found"] for o in observations)
        pose_count = sum(o["pose_ok"] for o in observations)
        fallback_count = sum(o["fallback_used"] for o in observations)
        report = pd.DataFrame([{
            "video_path": str(VIDEO_PATH),
            "label": VIDEO_LABEL,
            "status": "complete",
            "original_frames_reported": reported_total,
            "sampled_frames": len(observations),
            "detections_found": int(detection_count),
            "pose_successes": int(pose_count),
            "fallback_used_frames": int(fallback_count),
            "detection_rate": detection_count / max(len(observations), 1),
            "pose_success_rate": pose_count / max(len(observations), 1),
            "frame_step": FRAME_STEP,
            "final_npz": str(FINAL_NPZ),
            "frame_dir": str(FRAME_DIR),
            "elapsed_this_run_sec": round(time.time() - started, 1),
        }])
        report.to_csv(REPORT_CSV, index=False)
        print("\nVIDEO COMPLETE", flush=True)
        print("Sampled frames:", len(observations), flush=True)
        print("Detections found:", detection_count, "/", len(observations), flush=True)
        print("Pose successes:", pose_count, "/", len(observations), flush=True)
        print("Fallback used:", fallback_count, "frames", flush=True)
        print("Final NPZ:", FINAL_NPZ, flush=True)
        print("Report:", REPORT_CSV, flush=True)
    else:
        print("\nBATCH SAVED; VIDEO NOT COMPLETE YET", flush=True)
        print("New samples this run:", new_samples_this_run, flush=True)
        print("Total sampled frames saved:", len(observations), flush=True)
        print("Partial NPZ:", PARTIAL_NPZ, flush=True)
        print("Run this same cell again for the next batch.", flush=True)

Previously saved sampled frames: 35
Last saved original frame index: 136
FPS: 29.97002997002997 | reported total frames: 151
Processing original frame 140...
  detection=True; pose_ok=True; fallback=True; total_saved=36
Processing original frame 144...
  detection=True; pose_ok=True; fallback=True; total_saved=37
Processing original frame 148...
  detection=True; pose_ok=True; fallback=False; total_saved=38

VIDEO COMPLETE
Sampled frames: 38
Detections found: 38 / 38
Pose successes: 38 / 38
Fallback used: 15 frames
Final NPZ: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_single_tracked\biking_full_video_keypoints_v2.npz
Report: C:\Users\admin\ViTPose_Action_Recognition\data\processed\biking_full_video_report_v2.csv


## Cell 5 — Verify current saved output

This can be run after any batch, even before the full video is complete.

In [13]:
print("Frame JPEGs saved:", len(list(FRAME_DIR.glob("frame_*.jpg"))))
print("Partial NPZ exists:", PARTIAL_NPZ.is_file())
print("Final NPZ exists:", FINAL_NPZ.is_file())

inspect_path = FINAL_NPZ if FINAL_NPZ.is_file() else PARTIAL_NPZ
if inspect_path.is_file():
    with np.load(inspect_path, allow_pickle=False) as d:
        total = len(d["frame_indices"])
        print("Inspecting:", inspect_path.name)
        print("keypoints:", d["keypoints"].shape, "(sampled_frames, 17, 3)")
        print("boxes:", d["boxes"].shape)
        print("frames saved:", total)
        print("detections found:", int(d["detection_found"].sum()), "/", total)
        print("pose successes:", int(d["has_person"].sum()), "/", total)
        print("fallback used:", int(d["fallback_used"].sum()), "frames")
        print("first original frame indices:", d["frame_indices"][:10].tolist())
        print("last original frame index:", int(d["frame_indices"][-1]) if total else None)
        print("label:", d["label"].item())
else:
    print("No partial data yet. Run Cell 4.")

if REPORT_CSV.is_file():
    print("\nFinal report:")
    display(pd.read_csv(REPORT_CSV))

Frame JPEGs saved: 38
Partial NPZ exists: True
Final NPZ exists: True
Inspecting: biking_full_video_keypoints_v2.npz
keypoints: (38, 17, 3) (sampled_frames, 17, 3)
boxes: (38, 4)
frames saved: 38
detections found: 38 / 38
pose successes: 38 / 38
fallback used: 15 frames
first original frame indices: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36]
last original frame index: 148
label: Biking

Final report:


,video_path,label,status,original_frames_reported,sampled_frames,detections_found,pose_successes,fallback_used_frames,detection_rate,pose_success_rate,frame_step,final_npz,frame_dir,elapsed_this_run_sec
0,C:\Users\admin\ViTPose_Action_Recognition\data...,Biking,complete,151,38,38,38,15,1.0,1.0,4,C:\Users\admin\ViTPose_Action_Recognition\data...,C:\Users\admin\ViTPose_Action_Recognition\outp...,0.9


## Cell 6 — Notes and next steps

- Re-run **Cell 4 only** for the next batch until it reports `VIDEO COMPLETE`.
- If the kernel restarts, reopen the notebook, run Cells 1–4 in order, then run Cell 4 again. It resumes from the partial NPZ.
- Outputs are saved under new `_v2` names to preserve previous results.
- `detection_found` records whether YOLO returned a person box. `has_person` records whether pose inference returned usable finite joint coordinates. This avoids conflating detection success with pose success.
- Lowering confidence can increase recall but can also introduce false positives. Inspect saved JPEGs; a higher detection count alone is not enough.
- This notebook does not preserve identity across frames because ByteTrack is intentionally omitted for diagnosis. Once stable, the next step is to add tracking and then process a small, balanced subset of UCF101 classes.
- It does not train the LSTM. The LSTM comes after pose sequences from multiple labeled videos are validated.